In [1]:
import pandas as pd
from rdkit import Chem
from rdkit.Chem import MACCSkeys, Descriptors,AllChem
from rdkit.Chem.MolStandardize import rdMolStandardize
from matplotlib.colors import ListedColormap
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import GroupShuffleSplit
import rdkit.Chem.Descriptors3D
from morfeus import xtb, read_xyz
import os

In [2]:
file_path = './MixtureSolDB_with_pure_only_complete.csv'

solubility_df = pd.read_csv(file_path)

In [3]:
from rdkit import Chem
from rdkit.Chem import Descriptors
import pandas as pd
import numpy as np

def calc_rdkit_descriptors(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    
    desc = {
        'MolWt': Descriptors.MolWt(mol),
        'MolLogP': Descriptors.MolLogP(mol),
        'TPSA': Descriptors.TPSA(mol),
        'NumHDonors': Descriptors.NumHDonors(mol),
        'NumHAcceptors': Descriptors.NumHAcceptors(mol),
        'NumRotatableBonds': Descriptors.NumRotatableBonds(mol),
        'RingCount': Descriptors.RingCount(mol),
        'NumAromaticRings': Descriptors.NumAromaticRings(mol),
        'FractionCSP3': Descriptors.FractionCSP3(mol),
        'HeavyAtomCount': Descriptors.HeavyAtomCount(mol),
        'BertzCT': Descriptors.BertzCT(mol),
        'BalabanJ': Descriptors.BalabanJ(mol),
        'MaxPartialCharge': Descriptors.MaxPartialCharge(mol),
        'MinPartialCharge': Descriptors.MinPartialCharge(mol),
    }
    return desc

In [4]:
all_smiles = pd.concat([
    solubility_df['SMILES_Solute'],
    solubility_df['SMILES_Solvent1'],
    solubility_df['SMILES_Solvent2']
]).dropna().unique()

In [5]:
desc_list = []
valid_smiles = []

for sm in all_smiles:
    d = calc_rdkit_descriptors(sm)
    if d is not None:
        desc_list.append(d)
        valid_smiles.append(sm)

desc_df = pd.DataFrame(desc_list)
desc_df['SMILES'] = valid_smiles

In [6]:
# solute
solubility_df = solubility_df.merge(
    desc_df.add_prefix('solute_').rename(columns={'solute_SMILES': 'SMILES_Solute'}),
    on='SMILES_Solute',
    how='left'
)

# solvent1
solubility_df = solubility_df.merge(
    desc_df.add_prefix('solv1_').rename(columns={'solv1_SMILES': 'SMILES_Solvent1'}),
    on='SMILES_Solvent1',
    how='left'
)

# solvent2
solubility_df = solubility_df.merge(
    desc_df.add_prefix('solv2_').rename(columns={'solv2_SMILES': 'SMILES_Solvent2'}),
    on='SMILES_Solvent2',
    how='left'
)

In [8]:
row = solubility_df.iloc[0]
print(np.log(row['Solubility(mole_fraction)']))
print(np.log10(row['Solubility(mole_fraction)']))
print(row['LogS(mole_fraction)'])

-2.439550948067203
-1.0594835150674329
-1.0594835150674329


In [9]:
df = solubility_df.copy()

# 1. эмпирическое предсказание
df['mix_logS_linear'] = (
    df['x1'] * df['S1_pure_logS'] +
    df['x2'] * df['S2_pure_logS']
)

# 2. простые композиционные признаки
df['x1_minus_x2'] = df['x1'] - df['x2']
df['x1_times_x2'] = df['x1'] * df['x2']
df['x1_squared'] = df['x1'] ** 2
df['x2_squared'] = df['x2'] ** 2

# 3. нелинейные признаки на чистых растворимостях
df['pure_logS_diff'] = df['S1_pure_logS'] - df['S2_pure_logS']
df['pure_logS_abs_diff'] = (df['S1_pure_logS'] - df['S2_pure_logS']).abs()
df['x1_logS1'] = df['x1'] * df['S1_pure_logS']
df['x2_logS2'] = df['x2'] * df['S2_pure_logS']
df['x1x2_logS_diff'] = df['x1'] * df['x2'] * (df['S1_pure_logS'] - df['S2_pure_logS'])
df['x1x2_abs_logS_diff'] = df['x1'] * df['x2'] * (df['S1_pure_logS'] - df['S2_pure_logS']).abs()

# 4. mixture features по дескрипторам растворителей
# предполагается, что в df уже есть столбцы solv1_* и solv2_*

solv1_cols = [c for c in df.columns if c.startswith('solv1_')]
common_pairs = []

for c1 in solv1_cols:
    base = c1.replace('solv1_', '')
    c2 = f'solv2_{base}'
    if c2 in df.columns:
        common_pairs.append((c1, c2, base))

for c1, c2, base in common_pairs:
    df[f'mix_avg_{base}'] = df['x1'] * df[c1] + df['x2'] * df[c2]
    df[f'mix_diff_{base}'] = df[c1] - df[c2]
    df[f'mix_abs_diff_{base}'] = (df[c1] - df[c2]).abs()
    df[f'mix_x1x2_abs_diff_{base}'] = df['x1'] * df['x2'] * (df[c1] - df[c2]).abs()

In [10]:
df.to_csv("MixtureSolDB_with_features.csv", index=False)

print(df.shape)
print("Saved:")
print(" - MixtureSolDB_with_features.csv")

(117302, 127)
Saved:
 - MixtureSolDB_with_features.csv
